In [ ]:
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import part1 # to use code from tutorial
import torch

# Shared Step: Create Datasets with Three Representations

In [ ]:
# get the dataframe
df = pd.read_pickle("./merged_embeddings_sampled.pkl")

In [ ]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"

# train/test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
my_seeds = [0]
num_folds = 5

# (text, label) pairs, the format the tutorial code uses
def make_pairs(frame, column, label_names, label_column=LABEL_COL):
    to_id = {n: i for i, n in enumerate(label_names)}
    texts = frame[column].str.join(" ") if column in ("tokens", "lemmas") else frame[column]
    return list(zip(texts, [to_id[n] for n in frame[label_column]]))

# cross-validation folds, using the tutorial's function
def make_cv(column):
    pairs = make_pairs(train_df, column, label_names)
    return [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=s) for s in my_seeds]

cv_lemmas = make_cv("lemmas")        # for token indicator and bag of words
cv_sentences = make_cv("sentence")   # for the embeddings

In [ ]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [ ]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] = 1
        return vector

class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in x.split():
            index = self.vocab.get(item)
            if index is not None:
                vector[index] += 1
        return vector

# NOTE: I changed the embedding implementation to make it faster. It now builds a dictionary
# once instead of searching the whole dataframe on every call

class Embedding(Encoder):
    def __init__(self, embeddings_df):
        self.lookup = dict(zip(embeddings_df["sentence"], embeddings_df["embeddings"]))
        self.dim = len(next(iter(self.lookup.values())))

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
        return np.asarray(self.lookup[x], dtype=np.float32)

In [ ]:
# vocabulary from the train rows only (lemmas)
train_texts = [text for text, _ in make_pairs(train_df, "lemmas", label_names)]
vocab = get_vocab(train_texts, min_count=5, max_size=5000)

In [ ]:
# encoders

token_ind_encoder = TokenIndicator(vocab)
bow_encoder = BagOfWords(vocab)
sentence_em_encoder = Embedding(df)

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

Exp 1.B: Log-Lin with Bag of Words

Exp 1.C: Log-Lin with Sentence Embeddings

# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [ ]:
my_classifiers["mlp_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

In [21]:
for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

fold 0: accuracy 0.7082
fold 1: accuracy 0.7026
fold 2: accuracy 0.7047
fold 3: accuracy 0.7037
fold 4: accuracy 0.7043


Exp 2.B: MLP with Bag of Words

In [ ]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]
for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

Exp 2.C: MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp_embeddings-sentences"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_embeddings-sentences/seed-{seed}"
    my_classifiers["mlp_embeddings-sentences"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings-sentences"][0]["accuracies"]
for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

# Experiment 3: CNN

Exp 3.A: CNN with Token Indicator

Exp 3.B: CNN with Bag of Words

Exp 3.C: CNN with Sentence Embeddings